In [28]:
# Importing dependencies
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

In [29]:
# Data loading
df = pd.read_csv("quikr_car.csv")

print(df.head())

                                     name   company  year          Price  \
0    Hyundai Santro Xing XO eRLX Euro III   Hyundai  2007         80,000   
1                 Mahindra Jeep CL550 MDI  Mahindra  2006       4,25,000   
2              Maruti Suzuki Alto 800 Vxi    Maruti  2018  Ask For Price   
3  Hyundai Grand i10 Magna 1.2 Kappa VTVT   Hyundai  2014       3,25,000   
4        Ford EcoSport Titanium 1.5L TDCi      Ford  2014       5,75,000   

   kms_driven fuel_type  
0  45,000 kms    Petrol  
1      40 kms    Diesel  
2  22,000 kms    Petrol  
3  28,000 kms    Petrol  
4  36,000 kms    Diesel  


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        892 non-null    object
 1   company     892 non-null    object
 2   year        892 non-null    object
 3   Price       892 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: object(6)
memory usage: 41.9+ KB


In [31]:
# Checking for missing values
df.isnull().sum()

name           0
company        0
year           0
Price          0
kms_driven    52
fuel_type     55
dtype: int64

In [32]:
# Remove Invalid price rows
df = df[df['Price'] != 'Ask For Price']

In [33]:
df['Price'] = df['Price'].str.replace(',', '')
df['Price'] = df['Price'].astype(int)

In [34]:
df = df[df['year'].str.isnumeric()]
df['year'] = df['year'].astype(int)

In [35]:
df['kms_driven'] = df['kms_driven'].astype(str)

df['kms_driven'] = df['kms_driven'].str.replace(' kms','')

df['kms_driven'] = df['kms_driven'].str.replace(',', '')

In [36]:
df['kms_driven'] = pd.to_numeric(df['kms_driven'], errors = 'coerce')

In [37]:
#  Handling missing values
df['kms_driven'] = df['kms_driven'].fillna(df['kms_driven'].median())

In [38]:
df['fuel_type'] = df['fuel_type'].fillna(df['fuel_type'].mode()[0])

In [39]:
df["model"] = df["name"].apply(lambda x: x.split()[1])

In [40]:
df = df.reset_index(drop=True)

In [41]:
# Remove unrealistic values (outliers)
df = df[df['kms_driven'] < 200000]
df = df[df['Price'] < 6000000]

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 814 entries, 0 to 818
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        814 non-null    object 
 1   company     814 non-null    object 
 2   year        814 non-null    int32  
 3   Price       814 non-null    int32  
 4   kms_driven  814 non-null    float64
 5   fuel_type   814 non-null    object 
 6   model       814 non-null    object 
dtypes: float64(1), int32(2), object(4)
memory usage: 44.5+ KB


In [43]:
df.isnull().sum()

name          0
company       0
year          0
Price         0
kms_driven    0
fuel_type     0
model         0
dtype: int64

In [44]:
# Feature Selection
X = df[['company','model','year','kms_driven','fuel_type']]
y = df['Price']

In [45]:
print(X)

       company     model  year  kms_driven fuel_type
0      Hyundai    Santro  2007     45000.0    Petrol
1     Mahindra      Jeep  2006        40.0    Diesel
2      Hyundai     Grand  2014     28000.0    Petrol
3         Ford  EcoSport  2014     36000.0    Diesel
4         Ford      Figo  2012     41000.0    Diesel
..         ...       ...   ...         ...       ...
814     Toyota   Corolla  2009    132000.0    Petrol
815       Tata      Zest  2018     27000.0    Diesel
816   Mahindra    Quanto  2013     40000.0    Diesel
817      Honda     Amaze  2014     41000.0    Petrol
818  Chevrolet      Sail  2014     41000.0    Petrol

[814 rows x 5 columns]


In [46]:
print(y)

0       80000
1      425000
2      325000
3      575000
4      175000
        ...  
814    300000
815    260000
816    390000
817    180000
818    160000
Name: Price, Length: 814, dtype: int32


In [47]:
# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [48]:
# Encode categorical features
encoder = OneHotEncoder(handle_unknown='ignore')

column_transform = ColumnTransformer(
    [('encoder', encoder, ['company', 'model', 'fuel_type'])], remainder = 'passthrough'
)

In [49]:
model_lr = LinearRegression()

pipe_lr = Pipeline([
    ('preprocessor', column_transform),
    ('model', model_lr)
])

pipe_lr.fit(X_train, y_train)

C:\Users\nandi\AppData\Roaming\Python\Python312\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['company', 'model',
                                                   'fuel_type'])])),
                ('model', LinearRegression())])

In [50]:
y_pred_lr = pipe_lr.predict(X_test)

print("Linear regression R2 Score: ", r2_score(y_test, y_pred_lr))

Linear regression R2 Score:  0.8250741418089427


In [51]:
model_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)
pipe_rf = Pipeline([
    ('preprocessor', column_transform),
    ('model', model_rf)
])

pipe_rf.fit(X_train, y_train)

C:\Users\nandi\AppData\Roaming\Python\Python312\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['company', 'model',
                                                   'fuel_type'])])),
                ('model',
                 RandomForestRegressor(max_depth=10, min_samples_split=5,
                                       n_estimators=300, random_state=42))])

In [52]:
y_pred_rf = pipe_rf.predict(X_test)

print("Random Forest R2 Score:", r2_score(y_test, y_pred_rf))

Random Forest R2 Score: 0.6767942404867127


In [53]:
new_car = pd.DataFrame({
    'company':['Honda'],
    'model':['City'],
    'year':[2014],
    'kms_driven':[45000],
    'fuel_type': ['diesel']
})

predicted_price_lr = pipe_lr.predict(new_car)

print(f"Predicted Car Price by Linear Regression: ₹ {predicted_price_lr[0]:,.2f}")

Predicted Car Price by Linear Regression: ₹ 456,770.80


Two regression models were trained: Linear Regression and Random Forest Regressor.

To improve prediction accuracy, a new feature (car model) was extracted from the car name and added to the dataset. This feature helps the model distinguish between different car types from the same manufacturer.

After retraining with the updated features, Linear Regression achieved an R² score of 0.82, outperforming Random Forest (R² = 0.67). Therefore, Linear Regression was selected as the final model.


In [54]:
import pickle

pickle.dump(pipe_lr, open("car_price_model.pkl", "wb"))